In [3]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree  # <- quick mask tool [web:59]


# =========================
# PATHS / SETTINGS
# =========================
mapped_dir   = r"C:\\Users\\walten\\Desktop\\fyp\\Phase 3b Mapped results"
unmapped_dir = r"C:\\Users\\walten\\Desktop\\fyp\\lden"
output_dir   = r"C:\\Users\\walten\\Desktop\\fyp\\Lden_contours"
os.makedirs(output_dir, exist_ok=True)

dates = ['20230703', '20230704', '20230705', '20230706', '20230707', '20230708', '20230709']
hkiacentre = (22.308047, 113.918481)  # (lat, lon)

# Plot tuning
VMIN, VMAX = 60, 86
GRID_N = 600
SMOOTH_SIGMA = 1.0
FILL_STEP_DB = 2
THIN_LINE_STEP_DB = 2
BOLD_LINE_STEP_DB = 5
ZOOM_KM = 25


# =========================
# HELPERS
# =========================
def _std_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    if 'observer_id' in df.columns:
        df['observer_id'] = df['observer_id'].astype(str).str.strip()
    return df

def _logmean_db(arr_db: np.ndarray) -> float:
    arr_db = np.asarray(arr_db, dtype=float)
    e = 10 ** (arr_db / 10.0)
    return 10.0 * np.log10(np.mean(e))

def build_obs_coords(dates):
    coord = {}

    for date in dates:
        mapped_files = glob.glob(os.path.join(mapped_dir, f"*mapped*{date}*output*.csv"))
        unmapped_files = glob.glob(os.path.join(unmapped_dir, f"lden_unmapped_{date}*output*.csv"))
        for f in mapped_files + unmapped_files:
            if not os.path.exists(f):
                continue
            df = _std_cols(pd.read_csv(f))
            if not {'observer_id', 'lat', 'lon'}.issubset(df.columns):
                continue
            sub = df[['observer_id', 'lat', 'lon']].dropna()
            for oid, lat, lon in sub.itertuples(index=False, name=None):
                if oid not in coord:
                    coord[oid] = (float(lat), float(lon))

    obs_coords = pd.DataFrame({
        'observer_id': list(coord.keys()),
        'lat': [v[0] for v in coord.values()],
        'lon': [v[1] for v in coord.values()],
    })

    forced_coords = pd.DataFrame({
        'observer_id': ['1', '2', '3', '4', '1.0', '2.0', '3.0', '4.0', '01', '02', '03', '04'],
        'lat': [22.2839, 22.2845, 22.2851, 22.2857, 22.2839, 22.2845, 22.2851, 22.2857, 22.2839, 22.2845, 22.2851, 22.2857],
        'lon': [113.9170, 113.9180, 113.9190, 113.9200, 113.9170, 113.9180, 113.9190, 113.9200, 113.9170, 113.9180, 113.9190, 113.9200]
    })

    obs_coords = pd.concat([obs_coords, forced_coords], ignore_index=True)
    obs_coords['observer_id'] = obs_coords['observer_id'].astype(str).str.strip()
    obs_coords = obs_coords.drop_duplicates('observer_id').reset_index(drop=True)
    return obs_coords

def combine_day(date):
    mapped_files = glob.glob(os.path.join(mapped_dir, f"*mapped*{date}*output*.csv"))
    unmapped_files = glob.glob(os.path.join(unmapped_dir, f"lden_unmapped_{date}*output*.csv"))

    # ---- mapped ----
    mapped_list = []
    for f in mapped_files:
        df = _std_cols(pd.read_csv(f))
        if not {'observer_id', 'Lden_dba'}.issubset(df.columns):
            continue
        mapped_list.append(df[['observer_id', 'Lden_dba']].dropna())

    mapped = pd.concat(mapped_list, ignore_index=True) if mapped_list else pd.DataFrame(columns=['observer_id', 'Lden_dba'])
    if not mapped.empty:
        mapped = mapped.groupby('observer_id', as_index=False)['Lden_dba'].apply(lambda s: _logmean_db(s.to_numpy()))
        mapped = mapped.rename(columns={'Lden_dba': 'Lden_day_dBA'})
    else:
        mapped = pd.DataFrame(columns=['observer_id', 'Lden_day_dBA'])

    mapped_ids = set(mapped['observer_id'].tolist())

    # ---- unmapped ----
    unmapped_list = []
    for f in unmapped_files:
        df = _std_cols(pd.read_csv(f))
        if not {'observer_id', 'LDEN'}.issubset(df.columns):
            continue
        sub = df[['observer_id', 'LDEN']].dropna().rename(columns={'LDEN': 'Lden_day_dBA'})
        sub = sub[~sub['observer_id'].isin(mapped_ids)]
        unmapped_list.append(sub)

    unmapped = pd.concat(unmapped_list, ignore_index=True) if unmapped_list else pd.DataFrame(columns=['observer_id', 'Lden_day_dBA'])
    if not unmapped.empty:
        unmapped = unmapped.groupby('observer_id', as_index=False)['Lden_day_dBA'].apply(lambda s: _logmean_db(s.to_numpy()))

    day = pd.concat([mapped[['observer_id', 'Lden_day_dBA']], unmapped[['observer_id', 'Lden_day_dBA']]], ignore_index=True)
    return day

def plot_contour_xy(df_with_coords, value_col, title, png_name):
    # Project to local km around HKIA
    lat0, lon0 = hkiacentre
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(lat0))

    x = (df_with_coords['lon'].to_numpy() - lon0) * km_per_deg_lon
    y = (df_with_coords['lat'].to_numpy() - lat0) * km_per_deg_lat
    z = df_with_coords[value_col].to_numpy()

    xi = np.linspace(x.min(), x.max(), GRID_N)
    yi = np.linspace(y.min(), y.max(), GRID_N)
    XI, YI = np.meshgrid(xi, yi)

    pts = np.column_stack([x, y])
    ZI = griddata(pts, z, (XI, YI), method='linear')
    ZN = griddata(pts, z, (XI, YI), method='nearest')
    ZI = np.where(np.isnan(ZI), ZN, ZI)

    # ---- QUICK HACK: mask grid far from data using nearest distance [web:60][web:59]
    tree = cKDTree(pts)
    grid_pts = np.column_stack([XI.ravel(), YI.ravel()])
    d, _ = tree.query(grid_pts, k=1)
    D = d.reshape(XI.shape)

    nn_d, _ = tree.query(pts, k=2)          # nearest neighbor excluding self
    typ_spacing = np.median(nn_d[:, 1])
    maxdist = 2.5 * typ_spacing            # tune: 2.5–3.5
    w = np.clip(1 - (D / maxdist), 0, 1)   # weight 1 near data, 0 far away
    ZI = ZI * w + (np.nanmedian(ZI) * (1 - w))

    # -------------------------------------------------------------

    # Smooth without smearing NaNs outward too much
    med = np.nanmedian(ZI)
    Z_fill = np.where(np.isnan(ZI), med, ZI)
    Z_s = gaussian_filter(Z_fill, sigma=SMOOTH_SIGMA)
    Z_s[np.isnan(ZI)] = np.nan

    ZP = np.clip(Z_s, VMIN, VMAX)

    fill_levels = np.arange(VMIN, VMAX + 0.1, FILL_STEP_DB)

    fig, ax = plt.subplots(figsize=(7.2, 7.2), facecolor='white')

    cf = ax.contourf(XI, YI, ZP, levels=fill_levels, cmap='turbo', alpha=0.95)  # no extend tips
    cbar = plt.colorbar(cf, ax=ax, label='Lden (dBA)', shrink=0.9, pad=0.02)
    cbar.set_ticks(np.arange(VMIN, VMAX + 1, 5))

    # Thin lines (no labels)
    ax.contour(XI, YI, ZP, levels=np.arange(VMIN, VMAX + 0.1, THIN_LINE_STEP_DB),
               colors='k', linewidths=0.6, alpha=0.35)

    # Bold labeled lines every 5 dB
    cl = ax.contour(XI, YI, ZP, levels=np.arange(VMIN, VMAX + 0.1, BOLD_LINE_STEP_DB),
                    colors='k', linewidths=1.2, alpha=0.85)
    ax.clabel(cl, fmt='%d', fontsize=8)

    # HKIA
    ax.scatter(0, 0, c='black', s=130, marker='^',
               edgecolors='yellow', linewidth=2.0, label='HKIA', zorder=10)

    ax.set_title(title, fontsize=16, fontweight='bold', pad=10)
    ax.set_xlabel('X [km]')
    ax.set_ylabel('Y [km]')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.25, linestyle='--')
    ax.legend(loc='upper right', fontsize=10, frameon=True)

    ax.set_xlim(-ZOOM_KM, ZOOM_KM)
    ax.set_ylim(-ZOOM_KM, ZOOM_KM)

    outpath = os.path.join(output_dir, png_name)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Saved:", outpath)


# =========================
# MAIN
# =========================
print("=== PHASE 3B: FINAL COMBINE + CONTOURS (WITH MASK) ===")

obs_coords = build_obs_coords(dates)
print("✅ Coords:", len(obs_coords))

weekly_energy_sum = {}

for date in dates:
    print("\n📅 Date:", date)

    day = combine_day(date)
    if day.empty:
        print("  ⚪ No data")
        continue

    day_csv = os.path.join(output_dir, f"Lden_combined_{date}.csv")
    day.rename(columns={'Lden_day_dBA': 'Lden_combined_dBA'}).to_csv(day_csv, index=False)
    print("  ✅ Daily CSV:", day_csv, "rows:", len(day))

    dayc = day.merge(obs_coords, on='observer_id', how='left').dropna(subset=['lat', 'lon'])
    print("  ✅ With coords:", len(dayc))

    E = 10 ** (dayc['Lden_day_dBA'].to_numpy() / 10.0)
    for oid, e in zip(dayc['observer_id'].tolist(), E.tolist()):
        weekly_energy_sum[oid] = weekly_energy_sum.get(oid, 0.0) + float(e)

    plot_contour_xy(dayc, 'Lden_day_dBA', f"FYP Phase 3b: Daily Lden - {date}", f"Lden_contour_{date}.png")

if weekly_energy_sum:
    weekly = pd.DataFrame({
        'observer_id': list(weekly_energy_sum.keys()),
        'Lden_week_avg_dBA': [10.0 * np.log10(weekly_energy_sum[oid] / 7.0) for oid in weekly_energy_sum]
    })
    weekly_csv = os.path.join(output_dir, "Lden_weekly.csv")
    weekly.to_csv(weekly_csv, index=False)
    print("\n✅ Weekly CSV:", weekly_csv, "rows:", len(weekly))

    weeklyc = weekly.merge(obs_coords, on='observer_id', how='left').dropna(subset=['lat', 'lon'])
    plot_contour_xy(weeklyc, 'Lden_week_avg_dBA',
                    "FYP Phase 3b: Weekly Avg Daily Lden (Jul 3–9 2023)",
                    "Lden_contour_weekly.png")

print("\n🎉 DONE:", output_dir)


=== PHASE 3B: FINAL COMBINE + CONTOURS (WITH MASK) ===
✅ Coords: 18256

📅 Date: 20230703
  ✅ Daily CSV: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_combined_20230703.csv rows: 18248
  ✅ With coords: 18248
Saved: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_contour_20230703.png

📅 Date: 20230704
  ✅ Daily CSV: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_combined_20230704.csv rows: 18248
  ✅ With coords: 18248
Saved: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_contour_20230704.png

📅 Date: 20230705
  ✅ Daily CSV: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_combined_20230705.csv rows: 18248
  ✅ With coords: 18248
Saved: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_contour_20230705.png

📅 Date: 20230706
  ✅ Daily CSV: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_combined_20230706.csv rows: 18248
  ✅ With coords: 18248
Saved: C:\\Users\\walten\\Desktop\\fyp\\Lden_contours\Lden_contour_20230706.png

📅 Date: 20230707
  ✅ Daily CSV: C:\\User